# SurfaceTarget-CRC — phase 05 — patch 5 6

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.6
#
# RESEARCH PANEL AND EXPERIMENTAL VALIDATION ROADMAP
#
# OBJECTIVES:
# 1. Verify completed Phase 5.4 and 5.5 analyses
# 2. Preserve all 948 candidate genes
# 3. Confirm cross-scenario ranking agreement
# 4. Create a transparent 15-candidate review panel
# 5. Retain distinct biological targeting mechanisms
# 6. Audit protein, topology and functional evidence
# 7. Review all-tissue normal IHC observations
# 8. Construct candidate-specific validation priorities
# 9. Produce a human-readable research summary
# 10. Save provenance and a research checkpoint
#
# IMPORTANT:
# - Expert review panel, not an exhaustive target list
# - No therapeutic efficacy scores
# - No clinical safety predictions
# - No experimentally validated target claims
# - No previous data overwritten
# - No candidate genes excluded
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import hashlib
import json


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"
REPORTS = ROOT / "reports"
CHECKPOINTS = ROOT / "checkpoints"

for directory in [
    RESULTS,
    META,
    REPORTS,
    CHECKPOINTS
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


RANKING_PATH = (
    RESULTS /
    "surfacetarget_phase5_prioritization_v1.csv"
)

ALL_TISSUE_PATH = (
    RESULTS /
    "surfacetarget_phase5_all_tissue_evidence_audit.csv"
)

THREE_WAY_PATH = (
    RESULTS /
    "surfacetarget_phase5_three_scenario_top20.csv"
)

CORRELATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_scenario_rank_correlations.csv"
)

P54_PROVENANCE = (
    META /
    "surfacetarget_phase5_prioritization_provenance.json"
)

P55_PROVENANCE = (
    META /
    "surfacetarget_phase5_robustness_provenance.json"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.6")
print("RESEARCH PANEL AND VALIDATION ROADMAP")
print("=" * 65)


# ------------------------------------------------------------
# 2. VERIFY RESEARCH INPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


required_files = {
    "Phase 5.4 ranking": RANKING_PATH,
    "Phase 5.5 tissue audit": ALL_TISSUE_PATH,
    "Three-scenario candidates": THREE_WAY_PATH,
    "Rank correlations": CORRELATION_PATH,
    "Phase 5.4 provenance": P54_PROVENANCE,
    "Phase 5.5 provenance": P55_PROVENANCE
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} missing: {path}"
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(P54_PROVENANCE, "r") as handle:

    p54 = json.load(handle)


with open(P55_PROVENANCE, "r") as handle:

    p55 = json.load(handle)


ranking_hash = sha256_file(RANKING_PATH)


if ranking_hash != p54.get("output_sha256"):

    raise RuntimeError(
        "Phase 5.4 ranking checksum mismatch."
    )


if ranking_hash != p55.get(
    "ranking_source_sha256"
):

    raise RuntimeError(
        "Phase 5.5 source checksum mismatch."
    )


print("\nPhase 5.4 ranking checksum: PASSED")

print("Phase 5.5 provenance correspondence: PASSED")


# ------------------------------------------------------------
# 3. LOAD COMPLETED RESEARCH DATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING COMPLETE RESEARCH EVIDENCE")
print("-" * 65)


ranking = pd.read_csv(
    RANKING_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


evidence = pd.read_csv(
    ALL_TISSUE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


three_way = pd.read_csv(
    THREE_WAY_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


correlations = pd.read_csv(
    CORRELATION_PATH,
    index_col=0
)


if len(ranking) != 948:

    raise RuntimeError(
        "Unexpected ranking candidate count."
    )


if len(evidence) != 948:

    raise RuntimeError(
        "Unexpected evidence-audit candidate count."
    )


for name, table in {
    "Ranking": ranking,
    "Evidence": evidence
}.items():

    if table["gene_id"].duplicated().any():

        raise RuntimeError(
            f"{name} has duplicate gene IDs."
        )


if set(ranking["gene_id"]) != set(
    evidence["gene_id"]
):

    raise RuntimeError(
        "Candidate inventories disagree."
    )


# Patch 5.5 retained the Phase 5.4 columns.
# Verify exact correspondence rather than
# assuming that the tables are aligned.


ranking_index = ranking.set_index("gene_id")

evidence_index = evidence.set_index("gene_id")


original_fields = [
    column
    for column in ranking.columns
    if column != "gene_id"
]


if not ranking_index[
    original_fields
].equals(

    evidence_index.loc[
        ranking_index.index,
        original_fields
    ]

):

    raise RuntimeError(
        "Phase 5.4 and Phase 5.5 evidence differ."
    )


print("\nResearch candidates:", len(evidence))

print("Original ranking columns: VERIFIED")

print("Evidence reconciliation: PASSED")


# ------------------------------------------------------------
# 4. VERIFY THREE-SCENARIO AGREEMENT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RANKING ROBUSTNESS")
print("-" * 65)


SCENARIO_COLUMNS = [
    "phase5_rank_protein_first",
    "phase5_rank_normal_review_first",
    "phase5_rank_rna_discovery_first"
]


surface = evidence[
    evidence["phase5_followup_lane"].eq(
        "Putative cell-surface antigen review"
    )
].copy()


if len(surface) != 111:

    raise RuntimeError(
        "Expected 111 surface-review candidates."
    )


for column in SCENARIO_COLUMNS:

    ranks = pd.to_numeric(
        surface[column],
        errors="raise"
    ).astype(int)

    if set(ranks) != set(
        range(1, 112)
    ):

        raise RuntimeError(
            f"Invalid rank inventory: {column}"
        )


calculated_three_way = surface[

    surface[SCENARIO_COLUMNS]
    .le(20)
    .all(axis=1)

].copy()


if len(calculated_three_way) != 3:

    raise RuntimeError(
        "Expected three candidates in "
        "all scenario top-20 lists."
    )


if set(
    calculated_three_way["gene_id"]
) != set(
    three_way["gene_id"]
):

    raise RuntimeError(
        "Three-scenario membership differs "
        "from Patch 5.5."
    )


expected_three = {
    "CA9",
    "ADAM12",
    "NPC1L1"
}


if set(
    calculated_three_way["gene_name"]
) != expected_three:

    raise RuntimeError(
        "Unexpected three-scenario candidate identities."
    )


print("\nThree-scenario candidates:")

print(
    calculated_three_way[
        [
            "gene_name",
            *SCENARIO_COLUMNS
        ]
    ].to_string(index=False)
)


print("\nScenario rank correlations:")

print(
    correlations.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


print(
    "\nRanking scenarios are not independent "
    "biological validation experiments."
)


# ------------------------------------------------------------
# 5. DEFINE TRANSPARENT RESEARCH PANEL
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEFINING BIOLOGICAL RESEARCH PANEL")
print("-" * 65)


# This is a deliberately selected
# mechanistic review panel.
#
# Only Group A is chosen through the
# all-three-scenario ranking criterion.
#
# Groups B-E retain complementary
# pre-existing research hypotheses.
#
# Selection is not a claim of therapeutic
# superiority or an exhaustive search.


REVIEW_GROUPS = {

    "A | Cross-scenario surface hypotheses": [
        "CA9",
        "ADAM12",
        "NPC1L1"
    ],

    "B | Additional surface-antigen hypotheses": [
        "CDH3",
        "CLDN1",
        "DPEP1",
        "SLC7A5",
        "TDGF1",
        "LGR5"
    ],

    "C | Stromal-targeting hypothesis": [
        "FAP"
    ],

    "D | Membrane-accessibility investigations": [
        "SLC5A6",
        "CEMIP"
    ],

    "E | Extracellular microenvironment hypotheses": [
        "CTHRC1",
        "MXRA5",
        "SFRP4"
    ]

}


panel_records = []


for group, genes in REVIEW_GROUPS.items():

    for position, gene in enumerate(
        genes,
        start=1
    ):

        panel_records.append({

            "gene_name": gene,

            "phase56_review_group": group,

            "phase56_group_display_order":
                position
        })


panel_definition = pd.DataFrame(
    panel_records
)


if len(panel_definition) != 15:

    raise RuntimeError(
        "Expected 15 research-panel candidates."
    )


if panel_definition[
    "gene_name"
].duplicated().any():

    raise RuntimeError(
        "Duplicate gene in research panel."
    )


if evidence["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate original research gene symbols."
    )


panel = panel_definition.merge(

    evidence,

    on="gene_name",

    how="left",

    validate="one_to_one",

    indicator=True

)


if not panel["_merge"].eq("both").all():

    missing_genes = panel.loc[
        panel["_merge"].ne("both"),
        "gene_name"
    ].tolist()

    raise RuntimeError(
        "Panel genes missing from evidence: "
        f"{missing_genes}"
    )


panel = panel.drop(
    columns="_merge"
)


print("\nResearch-panel genes:", len(panel))

print("\nGroup distribution:")

print(
    panel[
        "phase56_review_group"
    ].value_counts(
        sort=False
    ).to_string()
)


# ------------------------------------------------------------
# 6. VALIDATE BIOLOGICAL LANE ASSIGNMENTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING BIOLOGICAL TARGETING GROUPS")
print("-" * 65)


EXPECTED_LANES = {

    "A": "Putative cell-surface antigen review",

    "B": "Putative cell-surface antigen review",

    "C": "Stromal-associated targeting review",

    "D": (
        "Membrane localization/"
        "accessibility verification"
    ),

    "E": "Extracellular microenvironment review"
}


for _, row in panel.iterrows():

    group_letter = row[
        "phase56_review_group"
    ][0]

    expected_lane = EXPECTED_LANES[
        group_letter
    ]

    actual_lane = row[
        "phase5_followup_lane"
    ]

    if actual_lane != expected_lane:

        raise RuntimeError(
            "Biological lane disagreement for "
            f"{row['gene_name']}: {actual_lane}"
        )


print("\nAll review-group assignments: PASSED")

print(
    "Stromal, membrane and extracellular "
    "mechanisms remain separate."
)


# ------------------------------------------------------------
# 7. VALIDATE PANEL EVIDENCE FIELDS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING PANEL EVIDENCE COMPLETENESS")
print("-" * 65)


required_panel_fields = {

    "gene_id",
    "ensembl_stable_id",
    "gene_name",

    "baseline_log2fc",
    "sensitivity_log2fc",
    "phase5_rna_conservative_log2fc",

    "protein_enriched",
    "phase5_protein_evidence_state",
    "mss_complete_pairs",

    "phase5_topology_class",
    "phase5_followup_lane",

    "phase5_normal_ihc_available",

    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",

    "phase55_normal_medium_high_tissue_names",
    "phase55_normal_high_tissue_names",

    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect",

    *SCENARIO_COLUMNS

}


missing_fields = (
    required_panel_fields
    - set(panel.columns)
)


if missing_fields:

    raise RuntimeError(
        "Panel evidence fields missing: "
        f"{sorted(missing_fields)}"
    )


print("\nRequired evidence fields: FOUND")

print("Panel gene identities: VALIDATED")


# ------------------------------------------------------------
# 8. PRESERVE NORMAL-TISSUE MISSINGNESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


def parse_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    allowed = {
        "true", "false", "1", "0",
        "", "nan", "none", "<na>"
    }

    unexpected = (
        values.notna()
        & ~values.isin(list(allowed))
    )

    if unexpected.any():

        raise RuntimeError(
            "Unexpected boolean values: "
            f"{values[unexpected].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


ihc_available = parse_bool(
    panel["phase5_normal_ihc_available"]
)


for column in [
    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high"
]:

    panel[column] = pd.to_numeric(
        panel[column],
        errors="coerce"
    )

    if panel.loc[
        ~ihc_available,
        column
    ].notna().any():

        raise RuntimeError(
            "Unobserved normal-IHC measurements "
            f"were assigned values: {column}"
        )


panel["phase56_normal_ihc_evidence_status"] = (
    np.where(
        ihc_available,

        "HPA normal-IHC observations available",

        "HPA normal-IHC observations unavailable"
    )
)


print(
    "\nPanel genes with HPA IHC:",
    int(ihc_available.sum())
)

print(
    "Panel genes without HPA IHC:",
    int((~ihc_available).sum())
)


print(
    "Missing normal expression has not "
    "been treated as low expression."
)


# ------------------------------------------------------------
# 9. PREPARE EXPERIMENTAL VALIDATION PLANS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING EXPERIMENTAL VALIDATION ROADMAP")
print("-" * 65)


def validation_plan(row):

    gene = row["gene_name"]

    group = row[
        "phase56_review_group"
    ][0]


    if group in ["A", "B"]:

        primary_question = (
            "Is the antigen expressed and "
            "accessible on malignant MSS CRC cells?"
        )

        primary_assay = (
            "Confirm expression in verified MSS CRC "
            "cell models using non-permeabilized "
            "flow cytometry or surface proteomics"
        )

        orthogonal_assay = (
            "Use single-cell/spatial expression "
            "analysis and matched normal-colon "
            "organoids to assess cell-type selectivity"
        )


    elif group == "C":

        primary_question = (
            "Is the protein enriched in "
            "tumor-associated fibroblasts?"
        )

        primary_assay = (
            "Perform multiplex immunostaining "
            "with fibroblast and epithelial markers"
        )

        orthogonal_assay = (
            "Evaluate spatial protein expression "
            "in tumor stroma and corresponding "
            "healthy tissue"
        )


    elif group == "D":

        primary_question = (
            "Does the protein present an "
            "accessible extracellular epitope?"
        )

        primary_assay = (
            "Measure live-cell extracellular "
            "antibody binding and plasma-membrane "
            "localization"
        )

        orthogonal_assay = (
            "Verify extracellular topology "
            "using orthogonal biochemical or "
            "surface-proteomics methods"
        )


    else:

        primary_question = (
            "Where is the protein deposited "
            "or retained in the tumor environment?"
        )

        primary_assay = (
            "Characterize extracellular matrix "
            "or secreted-protein localization "
            "in tumor sections"
        )

        orthogonal_assay = (
            "Compare tumor versus normal "
            "tissue localization using "
            "spatial protein measurements"
        )


    additional_questions = []


    protein_state = str(
        row["phase5_protein_evidence_state"]
    )


    if protein_state != (
        "Qualifying MSS paired-protein enrichment"
    ):

        additional_questions.append(
            "Obtain stronger tumor/normal "
            "protein-level evidence"
        )


    normal_available = str(
        row["phase56_normal_ihc_evidence_status"]
    )


    if "unavailable" in normal_available.lower():

        additional_questions.append(
            "Acquire independent normal-tissue "
            "expression measurements"
        )


    if gene in ["DPEP1", "SLC7A5"]:

        additional_questions.append(
            "Investigate binding to normal "
            "colon and kidney epithelia"
        )


    if gene in ["CDH3", "CLDN1"]:

        additional_questions.append(
            "Investigate normal epithelial "
            "expression and improve proteomic coverage"
        )


    if gene == "TDGF1":

        additional_questions.append(
            "Validate GPI-associated surface "
            "presentation and potential shedding"
        )


    if gene == "FAP":

        additional_questions.append(
            "Distinguish stromal antigen "
            "presentation from malignant-cell expression"
        )


    return {

        "primary_question": primary_question,

        "primary_assay": primary_assay,

        "orthogonal_assay": orthogonal_assay,

        "additional_review": (
            "; ".join(additional_questions)
            if additional_questions
            else "Standard specificity and validation review"
        )
    }


plan_records = []


for _, row in panel.iterrows():

    plan = validation_plan(row)

    plan_records.append({

        "gene_id": row["gene_id"],

        "gene_name": row["gene_name"],

        "review_group":
            row["phase56_review_group"],

        **plan

    })


validation = pd.DataFrame(
    plan_records
)


if len(validation) != 15:

    raise RuntimeError(
        "Experimental roadmap lost candidates."
    )


print("\nCandidate-specific plans:", len(validation))

print("Experimental roadmap: PREPARED")


# ------------------------------------------------------------
# 10. CREATE PANEL EVIDENCE SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REVIEWING PANEL EVIDENCE")
print("-" * 65)


panel_display_columns = [

    "gene_name",

    "phase56_review_group",

    "phase5_rna_conservative_log2fc",

    "phase5_protein_evidence_state",

    "phase5_topology_class",

    "phase55_normal_tissues_medium_high",

    "depmap_crc_median_gene_effect"

]


print("\n")

print(
    panel[
        panel_display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nImportant:")

print(
    "These genes belong to different "
    "biological targeting mechanisms."
)

print(
    "Their row order does not represent "
    "therapeutic superiority."
)


# ------------------------------------------------------------
# 11. PREPARE 948-GENE AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRESERVING COMPLETE CANDIDATE UNIVERSE")
print("-" * 65)


panel_lookup = panel.set_index(
    "gene_id"
)[
    "phase56_review_group"
].to_dict()


full_audit = evidence.copy(deep=True)


full_audit["phase56_in_review_panel"] = (
    full_audit["gene_id"].isin(
        set(panel["gene_id"])
    )
)


full_audit["phase56_review_group"] = (
    full_audit["gene_id"]
    .map(panel_lookup)
)


if len(full_audit) != 948:

    raise RuntimeError(
        "Complete candidate inventory changed."
    )


if int(
    full_audit["phase56_in_review_panel"].sum()
) != 15:

    raise RuntimeError(
        "Unexpected panel membership count."
    )


# Verify original evidence is unchanged.

if not evidence.equals(
    full_audit[evidence.columns]
):

    raise RuntimeError(
        "Original evidence was modified."
    )


print("\nComplete candidates preserved:", len(full_audit))

print("Selected for detailed panel:", 15)

print("Retained outside review panel:", 933)


# ------------------------------------------------------------
# 12. NORMAL-TISSUE DETAIL REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE REVIEW TABLE")
print("-" * 65)


normal_review_columns = [

    "gene_id",
    "gene_name",
    "phase56_review_group",

    "phase56_normal_ihc_evidence_status",

    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",

    "phase55_normal_medium_high_tissue_names",
    "phase55_normal_high_tissue_names"

]


normal_review = panel[
    normal_review_columns
].copy()


print("\nNormal-tissue review prepared.")

print(
    "Counts are descriptive observations, "
    "not clinical toxicity estimates."
)


# ------------------------------------------------------------
# 13. SAVE RESEARCH PANEL OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PANEL RESULTS")
print("-" * 65)


PANEL_PATH = (
    RESULTS /
    "surfacetarget_phase5_research_panel_v1.csv"
)

FULL_PATH = (
    RESULTS /
    "surfacetarget_phase5_full_universe_panel_audit.csv"
)

VALIDATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_experimental_validation_roadmap.csv"
)

NORMAL_PATH = (
    RESULTS /
    "surfacetarget_phase5_panel_normal_tissue_review.csv"
)


panel.to_csv(
    PANEL_PATH,
    index=False
)


full_audit.to_csv(
    FULL_PATH,
    index=False
)


validation.to_csv(
    VALIDATION_PATH,
    index=False
)


normal_review.to_csv(
    NORMAL_PATH,
    index=False
)


# Validate saved files.

saved_panel = pd.read_csv(
    PANEL_PATH,
    dtype={"gene_id": str}
)


saved_full = pd.read_csv(
    FULL_PATH,
    dtype={"gene_id": str}
)


if len(saved_panel) != 15:

    raise RuntimeError(
        "Saved research panel has incorrect size."
    )


if len(saved_full) != 948:

    raise RuntimeError(
        "Saved full audit has incorrect size."
    )


if set(saved_panel["gene_id"]) != set(
    panel["gene_id"]
):

    raise RuntimeError(
        "Saved panel identifiers changed."
    )


print("\nResearch tables: SAVED")

print("Output validation: PASSED")


# ------------------------------------------------------------
# 14. GENERATE HUMAN-READABLE RESEARCH SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH INTERPRETATION SUMMARY")
print("-" * 65)


REPORT_PATH = (
    REPORTS /
    "SurfaceTarget_CRC_Phase5_Research_Panel.md"
)


report_lines = [

    "# SurfaceTarget-CRC",

    "## Phase 5 — Research Panel and Validation Roadmap",

    "",

    (
        "This document summarizes an exploratory "
        "15-gene review panel derived from a larger "
        "948-gene cross-omics candidate universe."
    ),

    "",

    (
        "The panel is intended to guide research "
        "follow-up. It is not a validated therapeutic "
        "ranking or clinical recommendation."
    ),

    "",

    "## Research context",

    "",

    (
        "- TCGA: colorectal tumor-associated RNA "
        "enrichment, including computationally "
        "low-MSI discovery samples."
    ),

    (
        "- CPTAC: paired MSS colon tumor/normal "
        "proteomic evidence, subject to coverage limits."
    ),

    (
        "- HPA/UniProt: protein topology and "
        "descriptive healthy-tissue annotations."
    ),

    (
        "- DepMap: colorectal adenocarcinoma "
        "CRISPR fitness measurements; "
        "cell lines are not MSS-confirmed here."
    ),

    "",

    "## Prioritization limitations",

    "",

    (
        "- Protein-first and normal-review-first "
        "ranking scenarios are strongly correlated."
    ),

    (
        "- Only CA9, ADAM12 and NPC1L1 "
        "were in the top 20 of all three "
        "surface-review scenarios."
    ),

    (
        "- Cross-scenario agreement is not "
        "independent biological validation."
    ),

    (
        "- No candidate has been experimentally "
        "validated by this project."
    ),

    "",

    "## Candidate hypotheses",

    ""
]


for group, subset in panel.groupby(
    "phase56_review_group",
    sort=False
):

    report_lines.append(
        f"### {group}"
    )

    report_lines.append("")

    for _, row in subset.iterrows():

        gene = row["gene_name"]

        rna = row[
            "phase5_rna_conservative_log2fc"
        ]

        protein_state = row[
            "phase5_protein_evidence_state"
        ]

        topology = row[
            "phase5_topology_class"
        ]

        normal_n = row[
            "phase55_normal_tissues_medium_high"
        ]

        depmap_effect = row[
            "depmap_crc_median_gene_effect"
        ]


        normal_text = (
            "unavailable"
            if pd.isna(normal_n)
            else str(int(normal_n))
        )


        depmap_text = (
            "unavailable"
            if pd.isna(depmap_effect)
            else f"{depmap_effect:.3f}"
        )


        matching_plan = validation.loc[
            validation["gene_name"].eq(gene)
        ].iloc[0]


        report_lines.extend([

            f"**{gene}**",

            "",

            (
                f"- Conservative RNA log2FC: "
                f"{rna:.3f}"
            ),

            f"- CPTAC evidence: {protein_state}",

            f"- Topology: {topology}",

            (
                "- Normal tissues with medium/high "
                f"IHC: {normal_text}"
            ),

            (
                "- Median colorectal CRISPR "
                f"gene effect: {depmap_text}"
            ),

            (
                "- Primary validation question: "
                f"{matching_plan['primary_question']}"
            ),

            (
                "- Suggested first assay: "
                f"{matching_plan['primary_assay']}"
            ),

            ""
        ])


report_lines.extend([

    "## Scientific limitations",

    "",

    (
        "Normal-tissue IHC is not a direct "
        "measurement of therapeutic toxicity."
    ),

    "",

    (
        "Bulk tumor RNA and proteomics "
        "cannot establish malignant-cell-specific "
        "expression without additional evidence."
    ),

    "",

    (
        "Membrane annotation alone does not "
        "prove extracellular epitope accessibility "
        "or antibody internalization."
    ),

    "",

    (
        "CRISPR gene effects measure fitness "
        "following knockout and do not directly "
        "predict antibody-drug conjugate efficacy."
    ),

    "",

    (
        "Missing protein, IHC or functional "
        "measurements have not been interpreted "
        "as negative biological evidence."
    ),

    "",

    (
        "All 948 original candidate genes "
        "remain in the project dataset."
    ),

    ""
])


REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)


print("\nResearch summary created.")

print("Report path:")
print(REPORT_PATH)


# ------------------------------------------------------------
# 15. SAVE PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING FINAL PHASE 5 PANEL PROVENANCE")
print("-" * 65)


PANEL_PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_research_panel_provenance.json"
)


panel_provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.6",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_files": {

        name: str(path)
        for name, path in required_files.items()

    },

    "source_sha256": {

        name: sha256_file(path)
        for name, path in required_files.items()

    },

    "candidate_universe": 948,

    "research_panel_size": 15,

    "panel_selection": REVIEW_GROUPS,

    "cross_scenario_consistent_genes": sorted(
        expected_three
    ),

    "panel_method": (
        "Three-scenario ranking overlap "
        "combined with explicitly selected "
        "complementary biological hypotheses"
    ),

    "selection_limitations": (
        "The additional hypothesis groups "
        "are a transparent expert-review selection, "
        "not a data-derived optimal target ranking."
    ),

    "therapeutic_scores_assigned": False,

    "clinical_safety_inferred": False,

    "experimental_validation_completed": False,

    "all_candidate_genes_preserved": True,

    "panel_output": str(PANEL_PATH),

    "panel_sha256": sha256_file(PANEL_PATH),

    "full_audit_output": str(FULL_PATH),

    "full_audit_sha256": sha256_file(FULL_PATH),

    "report_output": str(REPORT_PATH),

    "report_sha256": sha256_file(REPORT_PATH),

    "limitations": [

        "Selection criteria reflect research "
        "follow-up decisions and are not "
        "clinical development criteria.",

        "Ranking scenarios are strongly correlated "
        "in two of the three comparisons.",

        "RNA evidence includes computationally "
        "low-MSI cases, not exclusively "
        "independently confirmed MSS cases.",

        "DepMap colorectal models are "
        "not confirmed MSS in this analysis.",

        "CPTAC measurements have gene-specific "
        "paired-sample coverage limitations.",

        "Tumor bulk measurements cannot "
        "assign expression to malignant cells.",

        "Normal-tissue IHC does not "
        "establish clinical safety.",

        "Surface topology does not "
        "demonstrate antibody accessibility.",

        "No new wet-lab experiments "
        "were performed by this project."

    ]
}


with open(
    PANEL_PROVENANCE_PATH,
    "w"
) as handle:

    json.dump(
        panel_provenance,
        handle,
        indent=4
    )


print("\nPanel provenance: SAVED")


# ------------------------------------------------------------
# 16. CREATE CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch56_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size / 1024**2
)


print("\nCheckpoint created.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.6 SUMMARY")
print("=" * 65)


print("\nResearch candidate universe:", 948)

print("Detailed research-panel candidates:", len(panel))

print(
    "Cross-scenario-consistent surface genes:",
    len(calculated_three_way)
)

print(
    "Biological research groups:",
    len(REVIEW_GROUPS)
)


print("\nSaved outputs:")

print("1. surfacetarget_phase5_research_panel_v1.csv")

print("2. surfacetarget_phase5_full_universe_panel_audit.csv")

print("3. surfacetarget_phase5_experimental_validation_roadmap.csv")

print("4. surfacetarget_phase5_panel_normal_tissue_review.csv")

print("5. SurfaceTarget_CRC_Phase5_Research_Panel.md")

print("6. surfacetarget_phase5_research_panel_provenance.json")


print("\nOriginal 948 candidates: PRESERVED")

print("Source rankings: UNCHANGED")

print("Normal-tissue missingness: PRESERVED")

print("No therapeutic efficacy scores assigned.")

print("No clinical safety claims made.")

print("No experimental target validation claimed.")


print("\nPhase 5 research checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nPATCH 5.6 COMPLETE")

print("=" * 65)